In [6]:
from datasets import load_dataset

# Load the complete dataset
dataset = load_dataset("louisbrulenaudet/clinical-trials", split="train")

print(f"Dataset size: {len(dataset)}")
print(f"Features: {dataset.features}")

Dataset size: 541897
Features: {'nct_id': Value('string'), 'updated_at': Value('timestamp[us]'), 'brief_title': Value('string'), 'official_title': Value('string'), 'acronym': Value('string'), 'study_type': Value('string'), 'overall_status': Value('string'), 'study_first_submit_date': Value('timestamp[ms]'), 'start_date': Value('timestamp[ms]'), 'primary_completion_date': Value('timestamp[ms]'), 'completion_date': Value('timestamp[ms]'), 'phases': List(Value('string')), 'enrollment_count': Value('float64'), 'minimum_age': Value('float64'), 'maximum_age': Value('float64'), 'sex': Value('string'), 'healthy_volunteers': Value('bool'), 'brief_summary': Value('string'), 'detailed_description': Value('string'), 'eligibility_criteria': Value('string'), 'lead_sponsor_name': Value('string'), 'lead_sponsor_class': Value('string'), 'org_study_id_info': {'id': Value('string'), 'link': Value('string'), 'type': Value('string')}, 'why_stopped': Value('string'), 'expanded_access_info': {'hasExpandedAcc

In [11]:
print("Number of features: ", len(dataset.features))

Number of features:  46


Our main features will be "detailed_description" and "brief_summary". Let's see how many rows are missing those features

In [8]:
print(len(dataset))
print("Without brief summary: ", dataset['brief_summary'].count(None))
print("Without detailed_description: ", dataset['detailed_description'].count(None))

541897
Without brief summary:  920
Without detailed_description:  175870


Looks like 1/3 of the data is missing the detailed description. A big number, but we consider that it still leaves us enough samples to work on. Let's look at some random samples:

In [10]:
import textwrap
import random
# --- Random rows peek ---
k = min(3, len(dataset))
idxs = random.sample(range(len(dataset)), k) if len(dataset) >= k else list(range(len(dataset)))
peek = dataset.select(idxs)
for i, row in enumerate(peek):
    print(f"\n--- Random row {i+1} (index={idxs[i]}) ---")
    # Print key fields for readability; you can expand this list as needed
    for key in [
        "nct_id", "brief_title", "study_type", "overall_status",
        "phases", "conditions", "minimum_age", "maximum_age", "sex",
        "brief_summary", "detailed_description"
    ]:
        if key in row:
            val = row[key]
            if isinstance(val, str) and len(val) > 500:
                val = textwrap.shorten(val, width=500, placeholder=" ...")
            print(f"{key}: {val}")


--- Random row 1 (index=158151) ---
nct_id: NCT02067520
brief_title: Intrathecal Hydromorphone for Cesarean Section
study_type: INTERVENTIONAL
overall_status: COMPLETED
phases: ['PHASE4']
conditions: ['Pain']
minimum_age: 18.0
maximum_age: None
sex: FEMALE
brief_summary: Introduction: Cesarean section (C/S) is usually performed under spinal with preservative free morphine for pain relief, but the investigators have a severe shortage of this formulation of morphine. Hydromorphone is a narcotic which acts peripherally and centrally to decrease pain. It has been used in spinals for postoperative pain relief and in pain pumps for relief of chronic pain. No randomized controlled studies have evaluated intrathecal (IT) hydromorphone for post C/S pain. Methods: ...
detailed_description: SpecificAims Cesarean section (C/S) is usually performed under spinal anesthesia with preservative free morphine added for pain relief, but currently the investigators have a severe shortage of this formulati

After looking through the dataset, we selected the features we found relevant and created a feature plan:

In [17]:
import pandas as pd

# --- Feature plan (keep/drop + justification) ---
plan_rows = [
    dict(field="nct_id", use="keep", role="identifier", justification="Unique trial ID for traceability/provenance."),
    dict(field="detailed_description", use="keep", role="primary_text", justification="Main source for simplification & summarization."),
    dict(field="brief_summary", use="keep", role="reference_text", justification="Comparison target for coverage & readability diffs."),
    dict(field="brief_title", use="keep", role="aux_text", justification="Concise signal of condition/intervention; useful for summaries."),
    dict(field="official_title", use="optional", role="aux_text", justification="Formal title; may provide specific terminology."),
    dict(field="study_type", use="keep", role="metadata", justification="Context for wording (interventional vs observational)."),
    dict(field="overall_status", use="keep", role="metadata", justification="Status cues phrasing and may affect inclusion."),
    dict(field="phases", use="keep", role="metadata", justification="Phase can inform lay explanation (e.g., late-stage confirmation)."),
    dict(field="conditions", use="keep", role="metadata", justification="Key to patient relevance; appears in the summary."),
    dict(field="minimum_age", use="keep", role="metadata", justification="Eligibility snippet (age bounds)."),
    dict(field="maximum_age", use="keep", role="metadata", justification="Eligibility snippet (age bounds)."),
    dict(field="sex", use="keep", role="metadata", justification="Eligibility snippet (sex)."),
    dict(field="healthy_volunteers", use="optional", role="metadata", justification="May be useful for eligibility sentences."),
    dict(field="eligibility_criteria", use="keep", role="aux_text", justification="Source for ‘for whom’ phrasing; optional extraction."),
    dict(field="lead_sponsor_name", use="optional", role="metadata", justification="Provenance; not typically in lay summary."),
    dict(field="lead_sponsor_class", use="drop", role="metadata", justification="Likely irrelevant for patient-facing text."),
    dict(field="enrollment_count", use="optional", role="metadata", justification="Contextual statistic; might be mentioned sparingly."),
    dict(field="start_date", use="optional", role="metadata", justification="Timeline context if needed."),
    dict(field="primary_completion_date", use="optional", role="metadata", justification="Timeline context if needed."),
    dict(field="completion_date", use="optional", role="metadata", justification="Timeline context if needed."),
    dict(field="keywords", use="optional", role="aux_text", justification="May seed domain vocabulary/jargon detection."),
    dict(field="locations", use="optional", role="metadata", justification="High-level geography; not always needed in summary."),
    dict(field="overall_officials", use="drop", role="metadata", justification="Low value for lay summary text."),
    dict(field="outcomes", use="keep", role="metadata", justification="Primary/secondary measures inform ‘what is measured’."),
    dict(field="design_info", use="keep", role="metadata", justification="Randomization/masking/purpose → plain-language explanation."),
    dict(field="mesh_terms", use="optional", role="metadata", justification="Useful for controlled vocabulary/jargon mapping."),
    dict(field="condition_browse_module", use="optional", role="metadata", justification="Taxonomy signs; optional for analysis."),
    dict(field="intervention_browse_module", use="optional", role="metadata", justification="Taxonomy signs; optional for analysis."),
    dict(field="why_stopped", use="optional", role="metadata", justification="If terminated, can inform a cautionary note in summary."),
    dict(field="expanded_access_info", use="drop", role="metadata", justification="Out-of-scope for lay summary generation."),
    dict(field="org_study_id_info", use="drop", role="metadata", justification="Administrative; not needed for text generation."),
    dict(field="arm_groups", use="optional", role="metadata", justification="If present, may help clarify groups in lay terms."),
    dict(field="interventions", use="keep", role="metadata", justification="Names/types of interventions: key to ‘what is tested’."),
    dict(field="study_references", use="drop", role="metadata", justification="Not needed for lay summary text."),
    dict(field="misc_info_module", use="drop", role="metadata", justification="Likely non-essential for our task."),
    dict(field="updated_at", use="optional", role="metadata", justification="Recency; can be used for filtering if needed."),
    dict(field="last_update_submit_qc_date", use="drop", role="metadata", justification="Quality control recency; drio for simplicity."),
    dict(field="last_update_post_date_struct", use="drop", role="metadata", justification="Administrative; drop for simplicity."),
    dict(field="study_first_post_date_struct", use="drop", role="metadata", justification="Administrative; drop for simplicity."),
    dict(field="study_first_submit_date", use="drop", role="metadata", justification="Administrative; drop for simplicity."),
    dict(field="std_ages", use="optional", role="metadata", justification="Age categories; redundant with min/max but handy."),
    dict(field="sampling_method", use="optional", role="metadata", justification="May inform wording but often technical."),
    dict(field="oversight_has_dmc", use="drop", role="metadata", justification="Oversight detail; too technical for lay summary."),
    dict(field="collaborators", use="drop", role="metadata", justification="Attribution; not needed in lay summary."),
    dict(field="acronym", use="drop", role="metadata", justification="Not relevant"),
    dict(field="study_population", use="keep", role="metadata", justification="Summary of the needed study population."),
]
plan_df = pd.DataFrame(plan_rows, columns=["field","use","role","justification"])
len(plan_df)
plan_df.to_csv("feature_plan.csv", index=False)

def df_to_markdown(df: pd.DataFrame) -> str:
    return df.to_markdown(index=False)

print("\n=== Feature Plan (Markdown) ===")
print(plan_df.to_markdown(index=False))



=== Feature Plan (Markdown) ===
| field                        | use      | role           | justification                                                     |
|:-----------------------------|:---------|:---------------|:------------------------------------------------------------------|
| nct_id                       | keep     | identifier     | Unique trial ID for traceability/provenance.                      |
| detailed_description         | keep     | primary_text   | Main source for simplification & summarization.                   |
| brief_summary                | keep     | reference_text | Comparison target for coverage & readability diffs.               |
| brief_title                  | keep     | aux_text       | Concise signal of condition/intervention; useful for summaries.   |
| official_title               | optional | aux_text       | Formal title; may provide specific terminology.                   |
| study_type                   | keep     | metadata       | Con